# 🫁 Medical AI Platform — Model Training
**EfficientNet-B0 fine-tuned on Chest X-Ray (Pneumonia) dataset**

Runtime: GPU (T4) · Expected time: ~20 minutes · Dataset: ~1.1 GB

After training, download `model.pth` and place it in the `weights/` folder of the project.

In [ ]:
# Step 1: Install kaggle and download dataset
!pip install kaggle -q

# Upload your kaggle.json API key when prompted
from google.colab import files
print('Upload your kaggle.json file:')
files.upload()

In [ ]:
import os
os.makedirs('/root/.kaggle', exist_ok=True)
!cp kaggle.json /root/.kaggle/
!chmod 600 /root/.kaggle/kaggle.json

!kaggle datasets download -d paultimothymooney/chest-xray-pneumonia -p /content/data --unzip
print('Dataset downloaded!')

In [ ]:
import torch
import torch.nn as nn
from torchvision import transforms, models, datasets
from torch.utils.data import DataLoader
import numpy as np
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

DATA_DIR = '/content/data/chest_xray'
BATCH_SIZE = 32
EPOCHS = 10
LR = 1e-4

In [ ]:
# Data transforms
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

train_ds = datasets.ImageFolder(os.path.join(DATA_DIR, 'train'), transform=train_transform)
val_ds   = datasets.ImageFolder(os.path.join(DATA_DIR, 'val'),   transform=val_transform)
test_ds  = datasets.ImageFolder(os.path.join(DATA_DIR, 'test'),  transform=val_transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f'Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}')
print(f'Classes: {train_ds.classes}')

In [ ]:
# Build model
model = models.efficientnet_b0(weights='IMAGENET1K_V1')
model.classifier[1] = nn.Linear(model.classifier[1].in_features, 2)
model = model.to(device)

# Class weights to handle imbalance
class_counts = np.bincount(train_ds.targets)
class_weights = 1.0 / class_counts
class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.5)

print('Model ready:', sum(p.numel() for p in model.parameters()), 'parameters')

In [ ]:
def evaluate(loader, model):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return correct / total * 100


best_val_acc = 0
history = {'train_loss': [], 'val_acc': []}

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0

    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    val_acc = evaluate(val_loader, model)
    avg_loss = running_loss / len(train_loader)
    history['train_loss'].append(avg_loss)
    history['val_acc'].append(val_acc)

    print(f'Epoch {epoch+1}/{EPOCHS} | Loss: {avg_loss:.4f} | Val Acc: {val_acc:.2f}%')

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'model.pth')
        print(f'  ✓ Saved best model (val_acc={val_acc:.2f}%)')

    scheduler.step()

print(f'\nBest Val Accuracy: {best_val_acc:.2f}%')

In [ ]:
# Test set evaluation
model.load_state_dict(torch.load('model.pth'))
test_acc = evaluate(test_loader, model)
print(f'Test Accuracy: {test_acc:.2f}%')

# Plot training curve
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history['train_loss'], marker='o')
ax1.set_title('Training Loss')
ax1.set_xlabel('Epoch')
ax2.plot(history['val_acc'], marker='o', color='green')
ax2.set_title('Validation Accuracy (%)')
ax2.set_xlabel('Epoch')
plt.tight_layout()
plt.savefig('training_curve.png', dpi=150)
plt.show()
print('Plot saved!')

In [ ]:
# Download model.pth to your local machine
from google.colab import files
files.download('model.pth')
files.download('training_curve.png')
print('Download started — place model.pth in the weights/ folder of the project.')